# IMQCAM File Collections: Builds, Parts, & Data Downloads

In the IMQCAM DMS:
- **Builds (Parent IGSNs)**: e.g. `CMXMAG00001` represents an entire manufacturing build.
- **Parts / Specimens (Child IGSNs)**: e.g. `CMXMAG00001-039` represents individual parts/specimens excised from that build.
- **Linked Metadata**: Material composition (`alloy718`, `Ti-6Al-4V`), AM parameters, and heat treatment schedules are attached to these IGSNs via form entries.
- **Related Data Files**: Measurement files (`ebsd_raw`, `ebsd_derived`, `optical_microscopy`, `ct_tomography`, etc.) are tagged with the part's `igsn` and `data_type`.

This notebook demonstrates how modelers can:
1. **Inspect all Builds & Parts** across the repository.
2. **Filter by Material / Composition** (e.g., `alloy718` / `IN718`, `Ti-6Al-4V`).
3. **Map Data Availability** (which parts have EBSD, microscopy, etc.).
4. **Query & Download Files** for target parts into structured local directories.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client, fetch_entries, entries_to_frame, extract_igsn

client = get_client()
print(f"Connected to {client.urlBase}")

## 1. Data Types: Total Files Counted vs. Queryable Items

- **`GET /imqcam/count`** counts physical files on the Girder storage server.
- **`GET /imqcam/datafiles`** queries Girder **Items** that carry specimen IGSN metadata.
- **Why they differ**:
  1. **Item vs. File granularity**: A single EBSD acquisition item often bundles multiple files (e.g., data scan + companion header/preview).
  2. **Metadata Readiness**: Files uploaded without complete sample indexing (`meta.igsn`, `experiment_date`) are counted on storage, but are not yet queryable via `/imqcam/datafiles` (e.g. `ct_tomography`).

In [ ]:
data_types = [d for d in client.get("imqcam/datatype") if d]
counts = client.get("imqcam/count")

rows = []
for d in data_types:
    files_counted = counts.get(d, 0)
    # Fetch retrievable items count with limit=1 to check availability quickly
    sample_batch = client.get("imqcam/datafiles", parameters={"dataType": d, "limit": 100})
    rows.append({
        "Data Type": d,
        "Files Counted (Server Storage)": files_counted,
        "Status": "Queryable via IGSN" if len(sample_batch) > 0 else "Pending IGSN Tagging"
    })

inventory_df = pd.DataFrame(rows).sort_values("Files Counted (Server Storage)", ascending=False).reset_index(drop=True)
inventory_df

## 2. Fetching & Flattening File Collections

`/imqcam/datafiles` caps page size at 100, so `fetch_datafiles()` pages through offset automatically. `datafiles_to_frame()` unpacks Girder metadata into a tabular DataFrame.

### Multi-IGSN handling

Some IMQCAM records can contain multiple comma-separated IGSNs in `meta.igsn`.

This notebook normalizes both:
- the IGSN(s) requested by the user, and
- the IGSN(s) stored in each record.

A file is returned when there is at least one matching constituent IGSN. This supports:
- a single IGSN,
- a comma-separated list of IGSNs, or
- a Python list of IGSNs.

The same normalization is applied before inventory and coverage summaries so multi-IGSN records do not hide valid parts.


In [ ]:
def split_igsns(value):
    """
    Normalize one or many IGSNs into a clean list.

    Supported inputs:
      - "CMXMAG00001-042"
      - "CMXMAG00001-042,CMXMAG00001-039"
      - ["CMXMAG00001-042", "CMXMAG00001-039"]

    This also handles stored DMS metadata where multiple IGSNs are kept
    in a comma-separated string.
    """
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        result = []
        for part in value:
            result.extend(split_igsns(part))
        return result

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none"}:
        return []

    return [
        part.strip()
        for part in text.split(",")
        if part.strip()
    ]


def fetch_datafiles(
    client,
    data_type,
    igsn=None,
    extra_fields=None,
    page_size=100,
):
    """
    Fetch all datafiles for a data type and optionally filter by one or many IGSNs.

    The IGSN filter is applied locally after pagination rather than through exact
    server-side equality. This prevents missing records whose meta.igsn contains
    multiple comma-separated constituent IGSNs.

    A record is kept when ANY requested IGSN overlaps with ANY stored IGSN.
    """
    params = {
        "dataType": data_type,
        "limit": page_size,
    }

    if extra_fields:
        params["extraFields"] = json.dumps(extra_fields)

    all_items = []
    offset = 0

    while True:
        params["offset"] = offset

        batch = client.get(
            "imqcam/datafiles",
            parameters=params,
        )

        all_items.extend(batch)

        print(
            f"Offset {offset}: "
            f"received {len(batch)} | "
            f"total collected {len(all_items)}"
        )

        if len(batch) < page_size:
            break

        offset += page_size

    if not igsn or igsn == "Any":
        return all_items

    requested_igsns = set(split_igsns(igsn))

    matched_items = []

    for item in all_items:
        stored_igsns = set(
            split_igsns(
                item.get("meta", {}).get("igsn")
            )
        )

        if requested_igsns & stored_igsns:
            matched_items.append(item)

    return matched_items

In [ ]:
def datafiles_to_frame(items):
    """Flatten Girder item envelopes and metadata into one row per item."""
    if not items:
        return pd.DataFrame()

    envelope = pd.DataFrame([
        {key: value for key, value in item.items() if key != "meta"}
        for item in items
    ])
    meta = pd.json_normalize(
        [item.get("meta", {}) for item in items],
        sep=".",
    )
    meta = meta.drop(
        columns=[column for column in meta.columns if column in envelope.columns],
        errors="ignore",
    )
    return pd.concat([envelope, meta], axis=1)

## 3. Inventory: Builds (Parents), Parts (Children), & Compositions

Every specimen cut from an additive build shares a parent IGSN root (e.g. build `CMXMAG00001` has child parts `CMXMAG00001-001`, `CMXMAG00001-039`, etc.).
We pull form entries to map all builds, their parts count, and their material composition (e.g., `alloy718`, `Ti-6Al-4V`).

In [ ]:
# Fetch all structured form entries
entries = fetch_entries(client, page_size=100)
entries_df = entries_to_frame(entries)

print(f"Retrieved {len(entries_df)} form entries")

In [ ]:
# Normalize to one row per constituent IGSN before inventory and coverage summaries.
valid_entries = entries_df[
    entries_df["igsn"].notna()
].copy()

# Preserve the original value for traceability.
valid_entries["igsn_raw"] = valid_entries["igsn"]

# Split single or comma-separated IGSN values.
valid_entries["igsn"] = valid_entries["igsn"].apply(split_igsns)

# Expand each constituent IGSN into its own row.
valid_entries = valid_entries.explode(
    "igsn",
    ignore_index=True,
)

valid_entries = valid_entries[
    valid_entries["igsn"].notna()
    & (valid_entries["igsn"].astype(str).str.len() > 0)
].copy()

valid_entries["build_igsn"] = valid_entries["igsn"].apply(
    lambda value:
        str(value).split("-")[0]
        if "-" in str(value)
        else str(value)
)

valid_entries["is_part"] = (
    valid_entries["igsn"]
    .astype(str)
    .str.contains("-")
)

print(f"Normalized IGSN rows: {len(valid_entries)}")
print(f"Distinct builds: {valid_entries['build_igsn'].nunique()}")
print(
    "Distinct parts:",
    valid_entries.loc[
        valid_entries["is_part"],
        "igsn",
    ].nunique(),
)

In [ ]:
# Extract material metadata and propagate build material to child parts.
# Keeping these steps together avoids relying on a previously executed cell.

def get_material(row):
    for column in [
        "alloy",
        "powderDetails.alloy",
        "material",
        "composition",
    ]:
        if column in row and pd.notna(row[column]):
            return str(row[column])

    # Fallback for older records where material is encoded in names/IDs.
    for column in ["buildName", "uniqueId"]:
        if column in row and pd.notna(row[column]):
            value = str(row[column])

            if "718" in value:
                return "alloy718"

            if "Ti64" in value or "Ti" in value:
                return "Ti-6Al-4V"

    return "Unspecified"


# Always create raw_material in this same cell before using it.
valid_entries["raw_material"] = valid_entries.apply(
    get_material,
    axis=1,
)

# Resolve one material per build and propagate it to child parts.
build_to_material = (
    valid_entries.loc[
        valid_entries["raw_material"] != "Unspecified"
    ]
    .groupby("build_igsn")["raw_material"]
    .first()
    .to_dict()
)

valid_entries["material"] = (
    valid_entries["build_igsn"]
    .map(build_to_material)
    .fillna("Unspecified")
)

valid_entries[
    [
        "igsn",
        "build_igsn",
        "raw_material",
        "material",
    ]
].drop_duplicates().head()


In [ ]:
def resolved_material_for_part(part_igsn):
    """
    Resolve a single verified material for a normalized part IGSN.

    Returns:
        material string when exactly one non-Unspecified material is found,
        otherwise None.
    """
    matches = (
        valid_entries.loc[
            (valid_entries["igsn"] == part_igsn)
            & (valid_entries["material"].notna())
            & (valid_entries["material"] != "Unspecified"),
            "material",
        ]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        print(
            f"Warning: multiple materials found for {part_igsn}: "
            f"{matches}. Material folder will not be inferred."
        )

    return None


In [ ]:
# Summarize builds, part counts, materials, and linked forms
build_summary = valid_entries.groupby("build_igsn").agg(
    total_parts=("igsn", lambda values: values[values.astype(str).str.contains("-")].nunique()),
    material=("material", "first"),
    forms_linked=("formName", lambda values: list(values.dropna().unique())),
).reset_index()

print(f"Registered builds: {len(build_summary)}")
build_summary

## 4. Query Data Files by Composition & Data Type

Filter by material (e.g., `alloy718`) or a specific build (`CMXMAG00001`), and query the measurement files (`ebsd_raw`, `ebsd_derived`, etc.) available for each part.

In [ ]:
# Configure the file query
target_datatype = "ebsd_raw" #use actual datatype ex: "ebsd_raw", "ebsd_derived"
target_material = "alloy718"  # Use "" for any material
target_part = "CMXMAG00001-042"                # Use an IGSN such as "CMXMAG00001-039", or "" for all parts

In [ ]:
# Fetch matching items
resolved_target_material = None

if target_part:
    print(f"Querying '{target_datatype}' for part {target_part}...")

    matched_files = fetch_datafiles(
        client,
        target_datatype,
        igsn=target_part,
    )

    # Resolve material from the selected part itself.
    resolved_target_material = resolved_material_for_part(target_part)

    if resolved_target_material:
        print(
            f"Resolved material for {target_part}: "
            f"{resolved_target_material}"
        )

        if (
            target_material
            and target_material != resolved_target_material
        ):
            print(
                f"Note: target_material='{target_material}' does not match "
                f"the resolved material '{resolved_target_material}'. "
                "The resolved material will be used for download paths."
            )
    else:
        print(
            f"Could not verify a unique material for {target_part}. "
            "Downloads will use 'unverified_material' as the folder."
        )

elif target_material:
    print(
        f"Querying '{target_datatype}' for material "
        f"{target_material}..."
    )

    all_datatype_items = fetch_datafiles(
        client,
        target_datatype,
    )

    all_datatype_df = datafiles_to_frame(
        all_datatype_items,
        explode_igsns=True,
    )

    material_parts = set(
        valid_entries.loc[
            (valid_entries["material"] == target_material)
            & valid_entries["is_part"],
            "igsn",
        ]
    )

    matched_ids = set(
        all_datatype_df.loc[
            all_datatype_df["igsn"].isin(material_parts),
            "_id",
        ]
    )

    matched_files = [
        item
        for item in all_datatype_items
        if item.get("_id") in matched_ids
    ]

    resolved_target_material = target_material

else:
    print(f"Querying all '{target_datatype}' items...")

    matched_files = fetch_datafiles(
        client,
        target_datatype,
    )

print(f"Retrieved {len(matched_files)} items")

In [ ]:
# Convert matching items into a DataFrame and inspect coverage
df_files = datafiles_to_frame(matched_files)

part_count = df_files["igsn"].nunique() if "igsn" in df_files.columns else 0
print(f"Found {len(df_files)} '{target_datatype}' items across {part_count} part(s)")

if len(df_files) and "igsn" in df_files.columns:
    print("\nItems per part:")
    print(df_files["igsn"].value_counts().to_string())

In [ ]:
# Preview file metadata
if len(df_files):
    columns_to_show = [
        column for column in ["name", "size", "data_type", "igsn", "created"]
        if column in df_files.columns
    ]
    display(df_files[columns_to_show].head(10))

## 5. Download Files Locally to Structured Directories

Save the measurement files organized by build and part folder:
`downloads/<material>/<build_igsn>/<part_igsn>/<filename>`
This structure allows external modelers, simulation meshes, and ML pipelines to locate input files by specimen ID without manual folder creation.

In [ ]:
# Configure a download preview
# Keep this False while testing; set True only after checking the result table.
download_files = False
download_limit = 3  # Use None to download every matched item

### Material-safe download paths

When a specific `target_part` is selected, the notebook now resolves the material from that part's metadata instead of trusting the manually entered `target_material`.

This prevents a valid part from being downloaded into the wrong material folder if the user changes `target_part` but forgets to update `target_material`.

If the part's material cannot be verified uniquely, the notebook uses:

`unverified_material`

rather than silently assigning an incorrect material.


In [ ]:
# Download the matched items
if download_files:
    items_to_download = matched_files

    if download_limit is not None:
        items_to_download = items_to_download[:download_limit]

    for item in items_to_download:
        constituent_igsns = split_igsns(
            item.get("meta", {}).get("igsn")
        )

        if target_part and target_part in constituent_igsns:
            item_igsn = target_part
        elif constituent_igsns:
            item_igsn = constituent_igsns[0]
        else:
            item_igsn = "unknown_part"

        build_igsn = (
            item_igsn.split("-")[0]
            if "-" in item_igsn
            else "general"
        )

        # Derive material from the selected/resolved part metadata.
        # Never fall back to a stale user-entered target_material for a part query.
        if target_part:
            material_folder = (
                resolved_target_material
                or resolved_material_for_part(item_igsn)
                or "unverified_material"
            )
        else:
            material_folder = (
                resolved_target_material
                or "unverified_material"
            )

        destination = (
            REPO_ROOT
            / "downloads"
            / target_datatype
            / material_folder
            / build_igsn
            / item_igsn
        )

        destination.mkdir(
            parents=True,
            exist_ok=True,
        )

        client.downloadItem(
            item["_id"],
            dest=str(destination),
        )

    print(f"Downloaded {len(items_to_download)} item(s)")

else:
    print(
        "Downloads are disabled. "
        "Set download_files = True to enable them."
    )

## 6. Requesting Custom Metadata with `extraFields` (e.g. `meta.composition`)

By default, `/imqcam/datafiles` projects only default envelope fields (`data_type`, `igsn`) for speed.
To retrieve custom metadata keys attached to items (such as `meta.composition = "IN718"`), pass `extraFields=json.dumps(["meta.composition"])`.
Once loaded into the DataFrame, the `composition` column is immediately available for filtering in pandas.

In [ ]:
# Fetch one specimen with custom item metadata
specimen_igsn = "CMXMAG00001-042"
raw_items = fetch_datafiles(
    client,
    data_type="ebsd_raw",
    igsn=specimen_igsn,
    extra_fields=["meta.composition"],
)
specimen_df = datafiles_to_frame(raw_items)

print(f"Retrieved {len(specimen_df)} items for {specimen_igsn}")
print("Available columns:", list(specimen_df.columns))

In [ ]:
# Filter the DataFrame by composition
target_comp = "718"
if "composition" in specimen_df.columns:
    comp_df = specimen_df[
        specimen_df["composition"].astype(str).str.contains(
            target_comp,
            case=False,
            na=False,
        )
    ]
    print(f"Found {len(comp_df)} items with composition matching '{target_comp}'")
    display(comp_df[["name", "size", "data_type", "igsn", "composition"]].head())
else:
    print("Composition metadata was not returned.")
    display(specimen_df[["name", "size", "data_type", "igsn"]].head())

In [ ]:
# Download the composition-filtered files
# Keep this False while testing; set it to True for the complete filtered collection.
download_filtered_files = False
download_limit = 3  # Set to an integer such as 3 for a small test

if download_filtered_files:
    filtered_ids = set(comp_df["_id"])
    filtered_items = [
        item for item in raw_items if item.get("_id") in filtered_ids
    ]
    if download_limit is not None:
        filtered_items = filtered_items[:download_limit]

    download_dir = (
        REPO_ROOT
        / "downloads"
        / "ebsd_raw"
        / specimen_igsn
        / target_comp
    )
    download_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {len(filtered_items)} filtered item(s) to {download_dir}...")
    for item in filtered_items:
        client.downloadItem(item["_id"], dest=str(download_dir))
    print(f"✅ Download complete: {download_dir}")
else:
    print("Downloads are disabled. Set download_filtered_files = True to enable them.")